# FlashSpec Colab A100 Profiling Runbook

先在 Colab 菜单里选择 `Runtime -> Change runtime type -> GPU`，如果要做正式 profiling，优先选择 A100。

推荐顺序：

1. 挂载 Drive，确认 GPU，安装项目和 Triton。
2. 检查 `ncu`，跑 correctness tests。
3. 先跑一次 sanity profiling，确认 `measured_*` 指标能正常产出。
4. 再跑矩阵化 profiling：先用 `dry` 看命令，再用 `small` 快速扫关键组合，最后按时间预算跑 `full`。
5. 汇总 JSON/manifest，生成 profile report。

默认项目目录是 `/content/drive/MyDrive/FlashSpecColab`，结果写到 `results/colab_kernels/` 和 `results/profile_matrix/`。

## 0. 挂载 Google Drive 并设置路径

结果会直接写到你的 Drive，避免 Colab runtime 重启后丢失。

In [ ]:
from google.colab import drive
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')
COLAB_RESULTS_DIR = PROJECT_DIR / 'results' / 'colab_kernels'
MATRIX_RESULTS_DIR = PROJECT_DIR / 'results' / 'profile_matrix'

drive.mount('/content/drive')
COLAB_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('COLAB_RESULTS_DIR:', COLAB_RESULTS_DIR)
print('MATRIX_RESULTS_DIR:', MATRIX_RESULTS_DIR)

## 1. 检查 GPU 环境

这里应该看到 `cuda available=True`。如果不是 A100，也可以先跑 correctness 和小规模 sanity，但 profiling 结果不要和 A100 结果直接比较。

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('当前 Colab runtime 没有 CUDA GPU，请切换到 GPU/A100 runtime。')

## 2. 克隆或更新项目

如果 Drive 里已经有仓库，会执行 `git pull --ff-only`。如果你在 notebook 里改了代码，先把改动提交或推送后再运行这一格。

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/honey-floria/FlashSpec.git', str(PROJECT_DIR)],
        check=True,
    )
else:
    raise RuntimeError(
        f'{PROJECT_DIR} 已存在但不是 Git 仓库，请改名或删除后重新 clone FlashSpec。'
    )

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
%pip install -e '.[triton]'

## 3. 检查 FlashSpec、CUDA、Triton 和 ncu

`HAS_TRITON=True` 才能跑 Triton kernel。`ncu` 用来采集 `measured_*` profiler 指标。

In [ ]:
import os
import shutil
import subprocess
import sys
import torch
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/FlashSpecColab')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from flashspec.triton_kernels import HAS_TRITON

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('HAS_TRITON:', HAS_TRITON)

NCU_BIN = shutil.which('ncu')
if NCU_BIN is None:
    for cand in ['/usr/local/cuda/bin/ncu', '/opt/nvidia/nsight-compute/ncu']:
        if os.path.exists(cand):
            NCU_BIN = cand
            break
if NCU_BIN is None:
    print('未找到 ncu，尝试 apt 安装 nsight-compute ...')
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'nsight-compute'], check=False)
    NCU_BIN = shutil.which('ncu') or '/usr/local/cuda/bin/ncu'

if os.path.exists(NCU_BIN):
    os.environ['PATH'] = os.path.dirname(NCU_BIN) + os.pathsep + os.environ.get('PATH', '')
    os.environ['NCU_BIN'] = NCU_BIN
    ver = subprocess.run([NCU_BIN, '--version'], capture_output=True, text=True)
    print('NCU_BIN:', NCU_BIN)
    print(ver.stdout.splitlines()[0] if ver.stdout else ver.stderr[:200])
else:
    os.environ['NCU_BIN'] = 'ncu'
    print('仍然找不到 ncu，后续带 --profile-ncu 的 cell 可能失败。')

## 4. 跑 correctness tests

A100 + Triton + CUDA 环境下先确认 Kernel 1 / Kernel 2 的输出仍然对齐 reference。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab
!python -m unittest discover -s tests

## 5. 跑 sanity profiling

先只跑两个代表点，确认 `--profile-ncu` 能输出带硬件指标的 JSON。这个 cell 成功后再进入矩阵化 profiling。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
OUT = REPO / 'results' / 'colab_kernels'
os.chdir(REPO)
OUT.mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get('NCU_BIN', 'ncu')

single_points = [
    ('triton_fused', 2048, 128),
    ('triton_paged', 2048, 128),
]

for backend, seq, hd in single_points:
    out = OUT / f'{backend}_s{seq}_d{hd}.json'
    cmd = [
        'python', 'benchmarks/microbench.py', '--backend', backend,
        '--batch', '16', '--heads', '32', '--seq-len', str(seq),
        '--head-dim', str(hd), '--block-size', '16',
        '--iters', '50', '--warmup', '10', '--repeats', '20',
        '--device', 'cuda', '--dtype', 'float16', '--json', '--include-raw',
        '--profile-ncu', '--ncu-bin', NCU_BIN,
        '--output', str(out),
    ]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## 6. 跑矩阵化 profiling

矩阵化 profiling 的意思是把关键参数组合成一张实验矩阵，然后逐项运行 benchmark + NCU，而不是手写很多单点 A/B cell。

- Kernel 1 `triton_fused`：扫描 `block_n / num_warps / num_splits / seq_len`。
- Kernel 2 `triton_paged`：扫描 `block_n / num_warps / length_pattern / paged_layout`。

运行建议：先把 `MATRIX_PRESET='dry'` 跑一遍，只打印命令不真正执行；确认没问题后改成 `small` 跑关键组合；Colab 时间充足时再改成 `full`。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
os.chdir(REPO)
NCU_BIN = os.environ.get('NCU_BIN', 'ncu')

# 取值：'dry' 只打印命令，'small' 跑关键组合，'full' 跑更完整矩阵。
MATRIX_PRESET = 'small'
PROFILE_NCU = True

common = [
    '--batch', '16', '--heads', '32', '--block-size', '16',
    '--iters', '50', '--warmup', '10', '--repeats', '20',
    '--device', 'cuda', '--dtype', 'float16',
]
if PROFILE_NCU and MATRIX_PRESET != 'dry':
    common += ['--profile-ncu', '--ncu-bin', NCU_BIN, '--ncu-launch-count', '5', '--ncu-timeout', '900']
if MATRIX_PRESET == 'dry':
    common += ['--dry-run']

if MATRIX_PRESET == 'full':
    fused_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '32,64,128', '--num-warps', '4,8',
        '--num-splits', 'auto,1,4,8', '--length-patterns', 'uniform',
    ]
    paged_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '32,64,128', '--num-warps', '4,8',
        '--length-patterns', 'uniform,descending',
        '--paged-layouts', 'contiguous,shuffled,interleaved',
    ]
else:
    # small：覆盖 block_n=64/128、Split-K 候选，以及 paged locality 的主要对照。
    fused_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '64,128', '--num-warps', '4',
        '--num-splits', 'auto,1,4,8', '--length-patterns', 'uniform',
    ]
    paged_args = [
        '--seq-lens', '2048,4096', '--head-dims', '128',
        '--block-ns', '64,128', '--num-warps', '4',
        '--length-patterns', 'uniform,descending',
        '--paged-layouts', 'contiguous,shuffled',
    ]

jobs = [
    ['python', 'scripts/profile_matrix.py', '--backend', 'triton_fused',
     '--output-dir', 'results/profile_matrix/fused'] + fused_args + common,
    ['python', 'scripts/profile_matrix.py', '--backend', 'triton_paged',
     '--output-dir', 'results/profile_matrix/paged'] + paged_args + common,
]

for cmd in jobs:
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

## 7. 汇总 JSON、图表和 manifest

`colab_kernels` 里的单点结果会生成对比图；`profile_matrix/*_manifest.csv` 会展示矩阵实验清单和每个组合的输出路径。

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
COLAB_RESULTS_DIR = REPO / 'results' / 'colab_kernels'
MATRIX_RESULTS_DIR = REPO / 'results' / 'profile_matrix'
ANALYSIS_DIR = COLAB_RESULTS_DIR / 'analysis'
os.chdir(REPO)

!python scripts/analyze_results.py --results-dir "{COLAB_RESULTS_DIR}" --output-dir "{ANALYSIS_DIR}"

summary = ANALYSIS_DIR / 'summary.csv'
if summary.exists():
    display(pd.read_csv(summary).head(30))

for manifest in sorted(MATRIX_RESULTS_DIR.glob('**/*_manifest.csv')):
    print('\nMANIFEST:', manifest)
    df = pd.read_csv(manifest)
    display(df.head(30))

for png in ['backend_comparison_s2048_d128.png', 'triton_scaling.png']:
    p = ANALYSIS_DIR / png
    if p.exists():
        display(Image(filename=str(p)))

## 8. 生成 profile report 和 source-line 命令

`profile_report.py` 会把 JSON 里的 profiler 指标整理成 Markdown，并打印 source-line / instruction attribution 命令。source-line 分析建议只挑 1-2 个关键 JSON 在 Colab 里跑。

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO = Path('/content/drive/MyDrive/FlashSpecColab')
os.chdir(REPO)

TARGET_JSON = Path('results/colab_kernels/triton_fused_s2048_d128.json')
if not TARGET_JSON.exists():
    candidates = sorted(Path('results/colab_kernels').glob('triton_*.json'))
    if not candidates:
        raise FileNotFoundError('找不到 triton_*.json，请先运行第 5 或第 6 节。')
    TARGET_JSON = candidates[0]

out_md = TARGET_JSON.with_name(TARGET_JSON.stem + '_profile.md')
subprocess.run(['python', 'scripts/profile_report.py', str(TARGET_JSON), '--output', str(out_md)], check=True)
print('profile report:', out_md)

data = json.loads(TARGET_JSON.read_text())
print('\nSource-line / instruction attribution command:')
print(data.get('nsight_compute_source_command', 'missing nsight_compute_source_command'))